# Retail Inventory & Sales Analysis

**Project 2 — Pandas + Matplotlib + Seaborn**

This project analyzes a retail inventory dataset to understand product performance, inventory levels, supplier performance, sales, and monthly trends.

**Dataset:** `retail_inventory.csv`
**Rows:** 2,000
**Columns:** 11


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("retail_inventory.csv")
df.head()


## 1. Initial Data Inspection

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.nunique()

In [ ]:
df.isna().sum()

In [ ]:
memory_usage_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f"Memory usage: {memory_usage_mb:.2f} MB")


## 2. Product Profile

In [ ]:
product_profile = (
    df.groupby(["product_name", "category"])
      .size()
      .reset_index(name="record_count")
      .sort_values("record_count", ascending=False)
)

product_profile.head(10)


In [ ]:
top_products_by_records = product_profile.nlargest(5, "record_count")
top_products_by_records


## 3. Inventory Summary

In [ ]:
inventory_summary = pd.Series({
    "Total units received": df["units_received"].sum(),
    "Total units sold": df["units_sold"].sum(),
    "Total units in stock": df["units_in_stock"].sum()
})

inventory_summary


In [ ]:
stock_by_product = (
    df.groupby(["product_id", "product_name", "category"])
      .agg(
          units_received=("units_received", "sum"),
          units_sold=("units_sold", "sum"),
          average_stock=("units_in_stock", "mean")
      )
      .reset_index()
)

stock_by_product["sell_through_rate"] = (
    stock_by_product["units_sold"] /
    stock_by_product["units_received"] * 100
)

stock_by_product.sort_values("units_sold", ascending=False).head(10)


## 4. Lowest Stock

In [ ]:
lowest_stock = df.loc[df["units_in_stock"].idxmin()]
lowest_stock


In [ ]:
low_stock_items = (
    df.loc[df["units_in_stock"] <= df["reorder_level"]]
      .sort_values("units_in_stock")
)

low_stock_items.head(10)


## 5. Product Sales Analysis

In [ ]:
sales_df = (
    df.assign(sales=df["units_sold"] * df["unit_price"])
      .groupby("product_name", as_index=False)
      .agg(
          units_sold=("units_sold", "sum"),
          sales=("sales", "sum"),
          unit_price=("unit_price", "first")
      )
      .sort_values("units_sold", ascending=False)
      .reset_index(drop=True)
)

sales_df


In [ ]:
total_revenue = sales_df["sales"].sum()
print(f"Total revenue: ₹{total_revenue:,.2f}")


In [ ]:
sales_df.sort_values("sales", ascending=False).head(10)


## 6. Supplier Analysis

In [ ]:
supplier_analysis = (
    df.assign(sales=df["units_sold"] * df["unit_price"])
      .groupby("supplier")
      .agg(
          units_received=("units_received", "sum"),
          units_sold=("units_sold", "sum"),
          average_unit_price=("unit_price", "mean"),
          sales_value=("sales", "sum")
      )
)

supplier_analysis["sell_through_rate"] = (
    supplier_analysis["units_sold"] /
    supplier_analysis["units_received"] * 100
)

supplier_analysis = (
    supplier_analysis
    .sort_values("sell_through_rate", ascending=False)
    .reset_index()
)

supplier_analysis


## 7. Supplier Performance Visualization

In [ ]:
sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data=supplier_analysis,
    x="sell_through_rate",
    y="supplier",
    hue="supplier",
    palette="viridis",
    legend=False
)

ax.bar_label(
    ax.containers[0],
    labels=[f"{value:.1f}%" for value in supplier_analysis["sell_through_rate"]],
    padding=4
)

plt.title("Supplier Sell-Through Rate", fontsize=16, fontweight="bold")
plt.xlabel("Sell-Through Rate (%)")
plt.ylabel("Supplier")
plt.xlim(0, supplier_analysis["sell_through_rate"].max() * 1.15)
plt.tight_layout()
plt.show()


## 8. Supplier Sales Share

In [ ]:
supplier_sales = (
    df.assign(sales=df["units_sold"] * df["unit_price"])
      .groupby("supplier")["sales"]
      .sum()
      .sort_values(ascending=False)
)

plt.figure(figsize=(8, 8))
plt.pie(
    supplier_sales,
    labels=supplier_sales.index,
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Supplier Share of Sales")
plt.tight_layout()
plt.show()


## 9. Product Sales Visualization

In [ ]:
top_sales = sales_df.sort_values("sales", ascending=False).head(10)

plt.figure(figsize=(11, 7))
ax = sns.barplot(
    data=top_sales,
    x="sales",
    y="product_name",
    hue="product_name",
    palette="magma",
    legend=False
)

ax.bar_label(
    ax.containers[0],
    labels=[f"₹{value:,.0f}" for value in top_sales["sales"]],
    padding=4
)

plt.title("Top 10 Products by Sales", fontsize=16, fontweight="bold")
plt.xlabel("Sales (₹)")
plt.ylabel("Product")
plt.tight_layout()
plt.show()


## 10. Monthly Inventory Trends

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.to_period("M")

monthly_inventory = (
    df.groupby("month")
      .agg(
          units_received=("units_received", "sum"),
          units_sold=("units_sold", "sum"),
          average_stock=("units_in_stock", "mean")
      )
)

monthly_inventory["net_units_change"] = (
    monthly_inventory["units_received"] -
    monthly_inventory["units_sold"]
)

monthly_inventory["stock_change"] = monthly_inventory["average_stock"].diff()
monthly_inventory["stock_change_pct"] = monthly_inventory["average_stock"].pct_change() * 100

monthly_inventory


In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(
    monthly_inventory.index.astype(str),
    monthly_inventory["units_received"],
    marker="o",
    label="Units Received"
)
plt.plot(
    monthly_inventory.index.astype(str),
    monthly_inventory["units_sold"],
    marker="o",
    label="Units Sold"
)

plt.title("Monthly Units Received vs Sold", fontsize=16, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Units")
plt.xticks(rotation=45)
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


## 11. August 2025 Sales Analysis

In [ ]:
august_df = df.loc[
    (df["date"] >= "2025-08-01") &
    (df["date"] < "2025-09-01")
].copy()

august_sales = (
    august_df.assign(sales=august_df["units_sold"] * august_df["unit_price"])
              .groupby("product_name", as_index=False)
              .agg(
                  units_sold=("units_sold", "sum"),
                  unit_price=("unit_price", "first"),
                  sales=("sales", "sum")
              )
              .sort_values("sales", ascending=False)
              .reset_index(drop=True)
)

august_sales


In [ ]:
plt.figure(figsize=(11, 8))
ax = sns.barplot(
    data=august_sales,
    x="sales",
    y="product_name",
    hue="product_name",
    palette="viridis",
    legend=False
)

ax.bar_label(
    ax.containers[0],
    labels=[f"₹{value:,.0f}" for value in august_sales["sales"]],
    padding=4,
    fontsize=9
)

plt.title("Product Sales — August 2025", fontsize=16, fontweight="bold")
plt.xlabel("Sales (₹)")
plt.ylabel("Product")
plt.tight_layout()
plt.show()


## 12. Category Performance

In [ ]:
category_analysis = (
    df.assign(sales=df["units_sold"] * df["unit_price"])
      .groupby("category")
      .agg(
          units_sold=("units_sold", "sum"),
          units_received=("units_received", "sum"),
          sales=("sales", "sum"),
          average_stock=("units_in_stock", "mean")
      )
)

category_analysis["sell_through_rate"] = (
    category_analysis["units_sold"] /
    category_analysis["units_received"] * 100
)

category_analysis.sort_values("sales", ascending=False)


In [ ]:
category_analysis["sales"].sort_values(ascending=False).plot(
    kind="bar",
    figsize=(10, 6),
    title="Sales by Category",
    xlabel="Category",
    ylabel="Sales (₹)"
)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 13. Store-Level Analysis

In [ ]:
store_analysis = (
    df.assign(sales=df["units_sold"] * df["unit_price"])
      .groupby("store")
      .agg(
          units_sold=("units_sold", "sum"),
          sales=("sales", "sum"),
          average_stock=("units_in_stock", "mean")
      )
      .sort_values("sales", ascending=False)
)

store_analysis


## 14. Useful Pandas Operations

In [ ]:
df["product_name"].value_counts().head(10)


In [ ]:
df.groupby("product_name")["units_sold"].sum().sort_values(ascending=False).head(10)


In [ ]:
pd.pivot_table(
    df,
    index="category",
    columns="store",
    values="units_sold",
    aggfunc="sum",
    fill_value=0
)


## Conclusion

This project combines Pandas data manipulation with Matplotlib and Seaborn visualizations.

### Main skills used

- Data inspection and validation
- `groupby()` and named aggregation
- `agg()`
- `value_counts()`
- `sort_values()`
- `nlargest()`
- `pivot_table()`
- Boolean filtering
- Date conversion and monthly grouping
- Derived metrics
- Matplotlib line and bar charts
- Seaborn bar plots
- Pie charts for proportional data
- Chart labels and formatting


